### Streaming Validation

**Kernel:** Select **StandAloneGuardValidation (.venv)** (top-right). After changing deps, run `uv sync` in this folder and restart the kernel.

**One-time setup** (run once in terminal, then restart kernel):

```bash
python -m guardrails_ai.competitor_check.post_install
```

This downloads the spaCy model `en_core_web_trf` that `CompetitorCheck` needs locally.

In [1]:
from guardrails import Guard
from guardrails_ai.competitor_check import CompetitorCheck

In [ ]:
COMPETITORS = ["OpenAI", "Anthropic", "Groq", "Gemini", "Claude", "Bard"]


def on_competitor_fail(value, fail_result):
    print(
        "This is a competitor check, please rephrase your response to avoid using any of the following competitors: "
        + ", ".join(COMPETITORS)
    )
    # Return the corrected text from the validator (or the original value as fallback)
    return fail_result.fix_value or fail_result.validated_chunk or value


textGuard = Guard().use(
    CompetitorCheck(
        competitors=COMPETITORS,
        on_fail=on_competitor_fail,
    )
)

**Normal Response**

In [ ]:
print(textGuard.validate("My name is Rahul Bisht and i works in google and use Gemini "))

**Streaming Response**

In [ ]:
stream = textGuard(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": "Compare OpenAI, Anthropic, and Google Gemini for building chatbots.",
        }
    ],
    stream=True
)
for chunk in stream:
    # Each chunk is a ValidationOutcome
    if chunk.validated_output:
        print(chunk.validated_output, end="", flush=True)
print()  # newline at end
print("Passed:", chunk.validation_passed)